# **1. Conectar Google Drive a Colab (Mounting)**


Como regla indispensable para trabajar con archivos locales en Colab, preparamos nuestro entorno de trabajo:
* `from google.colab import drive`: Llamamos a la herramienta de conexión.
* `drive.mount('/content/drive')`: Autorizamos el acceso a nuestro almacenamiento en la nube para buscar los archivos universitarios.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Cargar los datasets (Courses, Grades, Students)**


Importamos Pandas y cargamos de forma independiente los tres archivos CSV que componen nuestra base de datos relacional utilizando sus rutas exactas en el Drive.

In [3]:
import pandas as pd
courses = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/udmy/aplicada/COLLEGE - Courses.csv')
grades = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/udmy/aplicada/COLLEGE - Grades.csv')
students = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/udmy/aplicada/COLLEGE - Students.csv')

# **3. Explorar la estructura inicial**


Antes de realizar cualquier cruce, revisamos cómo están formadas las tablas individualmente:
* `courses`, `grades.head()`, `students.head()`: Imprimen los datos para identificar visualmente qué columnas comparten en común (como los identificadores numéricos).

In [4]:
courses
grades.head()
students.head()

,student_id,first_name,last_name,gender
0,2323,tj,adams,male
1,2326,christine,adams,NaN
2,2329,paul,fischer,male
3,2332,abby,badmus,female
4,2335,maria,jones,NaN


# **4. Unir tablas y aislar los mejores promedios**


Aplicamos la función de unión relacional:
* `pd.merge(students, grades, on = 'student_id')`: Conecta a los estudiantes con sus calificaciones utilizando la columna compartida `'student_id'`.
* Encadenamos esto con `.sort_values()` y `.head(10)` para crear una nueva variable (`df`) que aísla únicamente a los 10 mejores estudiantes.

In [5]:
pd.merge(students, grades, on = 'student_id').head()

df = pd.merge(students, grades, on = 'student_id').sort_values('grade', ascending = False).head(10)
df.head()

,student_id,first_name,last_name,gender,grade,course_id
19,2359,joanne,price,female,100,ECN433
9,2338,johnson,banky,male,98,CSC098
23,2368,melinda,smart,female,98,CSC220
25,2368,melinda,smart,female,98,ECN322
21,2365,rj,lara,male,96,CSC098


# **5. Búsqueda de registros específicos**


Aplicamos filtros booleanos sobre nuestra tabla cruzada para verificar si IDs específicos de estudiantes se encuentran en el ranking de excelencia:
* `df[df['student_id'] == 2332]`: Filtra la tabla para buscar al estudiante con ese número de identificación exacto.

In [6]:
df['student_id'] == 2332
df[df['student_id'] == 2332]
df[df['student_id'] == 2368]

,student_id,first_name,last_name,gender,grade,course_id
23,2368,melinda,smart,female,98,CSC220
25,2368,melinda,smart,female,98,ECN322


# **6. Unir tres tablas para crear una base de datos maestra**


Cuando necesitamos cruzar más de dos fuentes, encadenamos la función `merge`:
* `students_rec = students.merge(grades, on = 'student_id')`: Une estudiantes y notas.
* `students_rec1 = students_rec.merge(courses, on = 'course_id')`: A ese resultado le suma la tabla de cursos usando el identificador de la materia.

In [7]:
students_rec = students.merge(grades, on = 'student_id')
students_rec.head()

students_rec1 = students_rec.merge(courses, on = 'course_id')
students_rec1.head()

,student_id,first_name,last_name,gender,grade,course_id,course_name,semester
0,2323,tj,adams,male,85,CSC098,data visualization,summer
1,2323,tj,adams,male,90,DBA305,database administration,spring
2,2329,paul,fischer,male,90,CSC098,data visualization,summer
3,2329,paul,fischer,male,90,DBA305,database administration,spring
4,2329,paul,fischer,male,95,ECN322,cost benefit analysis,summer


# **7. Generar reportes académicos con GroupBy**


Una vez unida toda la información, podemos resumirla estadísticamente:
* `.groupby('course_name')`: Agrupa los registros por el nombre de la materia.
* `.size().sort_values()`: Cuenta el total de estudiantes matriculados por curso y los ordena.
* `students_records.size()`: Genera una agrupación multidimensional detallada por materia, apellidos, nombres y calificaciones.

In [8]:
reg_course = students_rec1.groupby('course_name')
reg_course.size().sort_values()

students_records = students_rec1.groupby(['course_name', 'last_name', 'first_name', 'grade'])
students_records.size()

course_name              last_name  first_name  grade
cost benefit analysis    andrew     melissa     90       1
                         banky      johnson     90       1
                         fischer    paul        95       1
                         smart      melinda     98       1
data analysis            banky      johnson     90       1
                         lara       rj          90       1
                         smart      melinda     98       1
data visualization       abubarka   fatima      85       1
                         adams      tj          85       1
                         banky      johnson     98       1
                         ceasar     romeo       80       1
                         fischer    paul        90       1
                         lara       rj          96       1
                         sylvester  tobias      90       1
database administration  abubarka   fatima      90       1
                         adams      tj          90       1
                         badmus     abby        93       1
                         fischer    paul        90       1
                         jones      maria       80       1
                         patterson  kevin       90       1
                         singh      pooja       87       1
                         smart      melinda     90       1
globalization            abubarka   fatima      95       1
                         anderson   abigael     80       1
                         brent      max         87       1
                         price      joanne      100      1
                         schmidts   paula       87       1
                         smart      melinda     90       1
introduction to biology  andrew     melissa     92       1
                         cooper     alice       95       1
vectors                  abubarka   fatima      90       1
                         anderson   abigael     82       1
                         badmus     abby        94       1
                         banky      johnson     85       1
                         drogba     george      87       1
                         fischer    paul        92       1
                         jones      christine   90       1
                         sylvester  tobias      80       1
dtype: int64

# **8. Unir tablas con nombres de columnas distintos**


Hasta ahora usábamos `on = '...'` porque la columna se llamaba exactamente igual en ambas tablas. Pero ¿qué pasa si en una tabla se llama `id` y en la otra se llama `student_id`?
* `grades.head()`: Inspeccionamos la tabla de notas.
* El código comentado `#pd.merge(..., left = 'id', right_on = 'student_id')` nos enseña la sintaxis oficial que debemos usar en Pandas cuando las llaves de enlace tienen nombres diferentes: especificamos de qué tabla viene cada nombre con `left_on` y `right_on`.

In [9]:
grades.head()
#pd.merge(students_rec1, grades, left_on = 'id', right_on = 'student_id')

,grade,student_id,course_id
0,85,2323,CSC098
1,90,2323,DBA305
2,90,2329,CSC098
3,90,2329,DBA305
4,95,2329,ECN322


# **9. Resumen del proceso completo**


En conclusión, este archivo nos enseña el manejo profesional de bases de datos relacionales:
* **Relaciones uno a muchos:** Aprendimos a conectar tablas independientes mediante identificadores comunes con `pd.merge()`.
* **Cadenas de unión:** Comprendimos cómo consolidar múltiples fuentes de datos (estudiantes, notas y cursos) en una sola tabla maestra.
* **Flexibilidad de nombres:** Descubrimos cómo resolver el problema cuando las columnas que sirven de puente tienen nombres distintos en cada archivo utilizando los parámetros específicos de enlace.